# Lab 19 — MITRE ATLAS & AI Threat Modeling

*AI Cybersecurity Masterclass · Lab 19 · AI60secLab*

**Companion video:** [▶ MITRE ATLAS: The AI-Attack Playbook](https://www.youtube.com/watch?v=QAUlOZvrlQY&list=PLG4l6Ssi9A00)

**Research question:** How does the threat model change when the protected system contains machine-learning or generative-AI components?

This lab downloads official MITRE ATLAS STIX data and converts AI-focused techniques into an analyzable table.

## Security foundation: threat modeling

**Threat modeling** asks four questions before an attacker does: what are we building, what can go wrong, what are we going to do about it, and did we do a good job?

Key ideas:
- **assets:** what is worth protecting. For an AI system that includes the training data, the model, the pipeline, and the outputs;
- **trust boundaries:** where data crosses from less trusted to more trusted (user input entering a model, a retrieved document entering a prompt);
- **adversaries:** who benefits from failure, and what they can touch.

Classic frameworks like STRIDE (spoofing, tampering, repudiation, information disclosure, denial of service, elevation of privilege) still apply to AI systems. MITRE ATLAS adds attacks specific to machine learning.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
from src.mitre_data import (
    ATLAS_STIX_URL,
    load_remote_or_local,
    technique_catalog,
)

local_fallback = DATA_DIR / "stix-atlas.json"

atlas_bundle = load_remote_or_local(
    ATLAS_STIX_URL,
    local_path=local_fallback,
)

atlas_techniques = technique_catalog(
    atlas_bundle,
    id_prefix="AML.",
)

print(
    "Active ATLAS techniques/sub-techniques:",
    len(atlas_techniques),
)
display(atlas_techniques.head())

Active ATLAS techniques/sub-techniques: 170


,stix_id,external_id,name,tactics,description,modified
0,attack-pattern--c02f812d-59cc-5366-b1aa-7eb051...,AML.T0000,Search Open Technical Databases,reconnaissance,Adversaries may search for publicly available ...,2025-04-09T00:00:00.000Z
1,attack-pattern--518338b9-9239-5e02-95f5-146bc7...,AML.T0000.000,Journals and Conference Proceedings,reconnaissance,Many of the publications accepted at premier a...,2025-04-09T00:00:00.000Z
2,attack-pattern--02ea7626-0eec-5a4b-98ff-b3f217...,AML.T0000.001,Pre-Print Repositories,reconnaissance,"Pre-Print repositories, such as arXiv, contain...",2021-05-13T00:00:00.000Z
3,attack-pattern--88a794e9-fa8c-5185-a677-bf476c...,AML.T0000.002,Technical Blogs,reconnaissance,Research labs at academic institutions and com...,2025-10-13T00:00:00.000Z
4,attack-pattern--4f36677b-3ba6-5556-9eba-0a2311...,AML.T0001,Search Open AI Vulnerability Analysis,reconnaissance,Much like the [Search Open Technical Databases...,2025-04-17T00:00:00.000Z


In [3]:
tactic_rows = atlas_techniques.copy()
tactic_rows["tactic"] = tactic_rows["tactics"].str.split(", ")
tactic_rows = tactic_rows.explode("tactic")
tactic_rows = tactic_rows[
    tactic_rows["tactic"].notna()
    & (tactic_rows["tactic"] != "")
]

tactic_counts = (
    tactic_rows.groupby("tactic", as_index=False)
    .agg(technique_count=("external_id", "nunique"))
    .sort_values("technique_count", ascending=False)
)

display(tactic_counts)

,tactic,technique_count
15,resource-development,26
9,impact,19
0,ai-attack-staging,17
5,defense-evasion,16
6,discovery,16
10,initial-access,15
12,persistence,14
7,execution,13
14,reconnaissance,12
8,exfiltration,9


In [4]:
terms = [
    "prompt",
    "poison",
    "model",
    "inference",
    "training",
    "supply chain",
]

for term in terms:
    mask = (
        atlas_techniques["name"].str.contains(
            term, case=False, na=False
        )
        | atlas_techniques["description"].str.contains(
            term, case=False, na=False
        )
    )

    matches = atlas_techniques.loc[
        mask,
        ["external_id", "name", "tactics"]
    ]

    print(f"\nTERM: {term} | matches: {len(matches)}")
    display(matches.head(10))


TERM: prompt | matches: 46


,external_id,name,tactics
8,AML.T0002.002,AI Agent Configuration,resource-development
30,AML.T0010.005,AI Agent Tool,initial-access
34,AML.T0011.002,Poisoned AI Agent Tool,execution
60,AML.T0034,Cost Harvesting,impact
62,AML.T0034.001,Resource-Intensive Queries,impact
63,AML.T0034.002,Agentic Resource Consumption,impact
87,AML.T0051,LLM Prompt Injection,execution
88,AML.T0051.000,Direct,execution
89,AML.T0051.001,Indirect,execution
90,AML.T0051.002,Triggered,execution



TERM: poison | matches: 24


,external_id,name,tactics
20,AML.T0008.002,Domains,resource-development
27,AML.T0010.002,Data,initial-access
30,AML.T0010.005,AI Agent Tool,initial-access
34,AML.T0011.002,Poisoned AI Agent Tool,execution
46,AML.T0018,Manipulate AI Model,"ai-attack-staging, persistence"
47,AML.T0018.000,Poison AI Model,"ai-attack-staging, persistence"
50,AML.T0019,Publish Poisoned Datasets,resource-development
51,AML.T0020,Poison Training Data,"persistence, resource-development"
63,AML.T0034.002,Agentic Resource Consumption,impact
75,AML.T0043.004,Insert Backdoor Trigger,ai-attack-staging



TERM: model | matches: 83


,external_id,name,tactics
0,AML.T0000,Search Open Technical Databases,reconnaissance
4,AML.T0001,Search Open AI Vulnerability Analysis,reconnaissance
5,AML.T0002,Acquire Public AI Artifacts,resource-development
7,AML.T0002.001,Models,resource-development
8,AML.T0002.002,AI Agent Configuration,resource-development
11,AML.T0005,Create Proxy AI Model,ai-attack-staging
12,AML.T0005.000,Train Proxy via Gathered AI Artifacts,ai-attack-staging
13,AML.T0005.001,Train Proxy via Replication,ai-attack-staging
14,AML.T0005.002,Use Pre-Trained Model,ai-attack-staging
16,AML.T0007,Discover AI Artifacts,discovery


... (2 outputs left out)


TERM: training | matches: 16


,external_id,name,tactics
5,AML.T0002,Acquire Public AI Artifacts,resource-development
7,AML.T0002.001,Models,resource-development
13,AML.T0005.001,Train Proxy via Replication,ai-attack-staging
16,AML.T0007,Discover AI Artifacts,discovery
27,AML.T0010.002,Data,initial-access
47,AML.T0018.000,Poison AI Model,"ai-attack-staging, persistence"
50,AML.T0019,Publish Poisoned Datasets,resource-development
51,AML.T0020,Poison Training Data,"persistence, resource-development"
53,AML.T0024,Exfiltration via AI Inference API,exfiltration
54,AML.T0024.000,Infer Training Data Membership,exfiltration



TERM: supply chain | matches: 14


,external_id,name,tactics
24,AML.T0010,AI Supply Chain Compromise,initial-access
25,AML.T0010.000,Hardware,initial-access
26,AML.T0010.001,AI Software,initial-access
27,AML.T0010.002,Data,initial-access
29,AML.T0010.004,Container Registry,initial-access
30,AML.T0010.005,AI Agent Tool,initial-access
31,AML.T0011,User Execution,execution
32,AML.T0011.000,Unsafe AI Artifacts,execution
33,AML.T0011.001,Malicious Package,execution
50,AML.T0019,Publish Poisoned Datasets,resource-development


## Threat-model exercise

Choose one system:
- security classifier;
- RAG assistant;
- autonomous AI agent;
- model-training pipeline.

Select five relevant ATLAS techniques and document:

**Asset → Technique → Preconditions → Security impact → Observable signal → Mitigation hypothesis**

## From lab to SOC

Threat models turn into concrete work: controls with owners, detections for the risky boundaries, and red-team tests (Lab 27). For AI systems the human decision after the model is part of the attack surface: an overtrusted output can cause the damage itself.

**Check yourself**
1. List three trust boundaries in a system where an LLM summarizes threat intelligence from the internet.
2. Map one ATLAS-style AI attack to a STRIDE category.
3. Why is "the analyst trusts the model too much" a security risk?